# **Machine Learning Kelompok Jet Tempur**

Notebook ini digunakan untuk membandingkan model Machine Learning **tanpa fuzzy** dan **dengan fuzzy score**.

Eksperimen yang dijalankan:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

Setiap algoritma diuji dalam dua skenario:

- **Tanpa Fuzzy**: menggunakan 5 fitur asli.
- **dengan Fuzzy**: menggunakan 5 fitur asli + `mamdani_score` + `sugeno_score`.

Catatan:
- Tidak dilakukan balancing/undersampling karena distribusi tidak terlalu timpang.
- Tidak dilakukan scaling karena fitur numerik utama berada pada rentang yang seragam.

## **0. Import Library**

In [1]:
!pip install -q xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

## **1. Load Dataset**

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train_hybrid = pd.read_csv("train_hybrid.csv")
test_hybrid = pd.read_csv("test_hybrid.csv")

print("Train shape        :", train.shape)
print("Test shape         :", test.shape)
print("Train hybrid shape :", train_hybrid.shape)
print("Test hybrid shape  :", test_hybrid.shape)

display(train.head())
print()
print("-" * 139)
display(train_hybrid.head())

Train shape        : (103904, 6)
Test shape         : (25976, 6)
Train hybrid shape : (103904, 8)
Test hybrid shape  : (25976, 8)


,Type of Travel,Class,Inflight wifi service,Online boarding,Inflight entertainment,satisfaction
0,Personal Travel,Eco Plus,3,3,5,neutral or dissatisfied
1,Business travel,Business,3,3,1,neutral or dissatisfied
2,Business travel,Business,2,5,5,satisfied
3,Business travel,Business,2,2,2,neutral or dissatisfied
4,Business travel,Business,3,5,3,satisfied



-------------------------------------------------------------------------------------------------------------------------------------------


,Type of Travel,Class,Inflight wifi service,Online boarding,Inflight entertainment,satisfaction,mamdani_score,sugeno_score
0,Personal Travel,Eco Plus,3,3,5,neutral or dissatisfied,51.052953,60.714286
1,Business travel,Business,3,3,1,neutral or dissatisfied,51.052953,58.333333
2,Business travel,Business,2,5,5,satisfied,83.666667,75.000000
3,Business travel,Business,2,2,2,neutral or dissatisfied,48.947047,45.000000
4,Business travel,Business,3,5,3,satisfied,83.666667,75.000000


## **2. Validasi Dataset**

In [3]:
print("Missing value train:")
display(train.isnull().sum())
print("-" * 24)
print("Missing value test:")
display(test.isnull().sum())
print("-" * 24)
print("Missing value train_hybrid:")
display(train_hybrid.isnull().sum())
print("-" * 24)
print("Missing value test_hybrid:")
display(test_hybrid.isnull().sum())
print("-" * 32)
print("Distribusi target train:")
display(train["satisfaction"].value_counts(normalize=True).mul(100).round(2))
print("-" * 32)
print("Distribusi target test:")
display(test["satisfaction"].value_counts(normalize=True).mul(100).round(2))

Missing value train:


,0
Type of Travel,0
Class,0
Inflight wifi service,0
Online boarding,0
Inflight entertainment,0
satisfaction,0


------------------------
Missing value test:


,0
Type of Travel,0
Class,0
Inflight wifi service,0
Online boarding,0
Inflight entertainment,0
satisfaction,0


------------------------
Missing value train_hybrid:


...

,0
Type of Travel,0
Class,0
Inflight wifi service,0
Online boarding,0
Inflight entertainment,0
satisfaction,0
mamdani_score,0
sugeno_score,0


--------------------------------
Distribusi target train:


,proportion
satisfaction,
neutral or dissatisfied,56.67
satisfied,43.33


--------------------------------
Distribusi target test:


,proportion
satisfaction,
neutral or dissatisfied,56.1
satisfied,43.9


## **3. Konfigurasi Fitur dan Target**

In [4]:
TARGET = "satisfaction"

FEATURES_ORIGINAL = [
    "Type of Travel",
    "Class",
    "Inflight wifi service",
    "Online boarding",
    "Inflight entertainment"
]

FEATURES_HYBRID = [
    "Type of Travel",
    "Class",
    "Inflight wifi service",
    "Online boarding",
    "Inflight entertainment",
    "mamdani_score",
    "sugeno_score"
]

CATEGORICAL_FEATURES = ["Type of Travel", "Class"]

print("Original features:", FEATURES_ORIGINAL)
print("Hybrid features  :", FEATURES_HYBRID)

Original features: ['Type of Travel', 'Class', 'Inflight wifi service', 'Online boarding', 'Inflight entertainment']
Hybrid features  : ['Type of Travel', 'Class', 'Inflight wifi service', 'Online boarding', 'Inflight entertainment', 'mamdani_score', 'sugeno_score']


## **4. Encode Label Target**

In [5]:
label_encoder = LabelEncoder()

y_train_original = label_encoder.fit_transform(train[TARGET])
y_test_original = label_encoder.transform(test[TARGET])

y_train_hybrid = label_encoder.transform(train_hybrid[TARGET])
y_test_hybrid = label_encoder.transform(test_hybrid[TARGET])

print("Label mapping:")
for label, encoded in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{label} -> {encoded}")

Label mapping:
neutral or dissatisfied -> 0
satisfied -> 1


## **5. Siapkan Data Original dan Hybrid**

In [6]:
X_train_original = train[FEATURES_ORIGINAL]
X_test_original = test[FEATURES_ORIGINAL]

X_train_hybrid = train_hybrid[FEATURES_HYBRID]
X_test_hybrid = test_hybrid[FEATURES_HYBRID]

print("X_train_original :", X_train_original.shape)
print("X_test_original  :", X_test_original.shape)
print("X_train_hybrid   :", X_train_hybrid.shape)
print("X_test_hybrid    :", X_test_hybrid.shape)

X_train_original : (103904, 5)
X_test_original  : (25976, 5)
X_train_hybrid   : (103904, 7)
X_test_hybrid    : (25976, 7)


## **6. Logistic Regression**

### **6.1 Logistic Regression Tanpa Fuzzy**

In [7]:
lr_original_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

lr_original_pipeline.fit(X_train_original, y_train_original)
y_pred_lr_original = lr_original_pipeline.predict(X_test_original)

metrics_lr_original = {
    "Model": "Logistic Regression Without Fuzzy",
    "Accuracy": accuracy_score(y_test_original, y_pred_lr_original),
    "Precision": precision_score(y_test_original, y_pred_lr_original),
    "Recall": recall_score(y_test_original, y_pred_lr_original),
    "F1 Score": f1_score(y_test_original, y_pred_lr_original)
}

### **6.2 Visualisasi Logistic Regression**

xxxxx

### **6.3 Hasil Logistic Regression Tanpa Fuzzy**

In [8]:
print("=============== Logistic Regression Without Fuzzy ===============")

print("\nAccuracy  :", round(metrics_lr_original["Accuracy"], 4))
print("Precision :", round(metrics_lr_original["Precision"], 4))
print("Recall    :", round(metrics_lr_original["Recall"], 4))
print("F1 Score  :", round(metrics_lr_original["F1 Score"], 4))

print("\nClassification Report:")
print(classification_report(
    y_test_original,
    y_pred_lr_original,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_original, y_pred_lr_original))

display(pd.DataFrame([metrics_lr_original]))

=============== Logistic Regression Without Fuzzy ===============

Accuracy  : 0.8407
Precision : 0.8157
Recall    : 0.823
F1 Score  : 0.8194

Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.86      0.85      0.86     14573
              satisfied       0.82      0.82      0.82     11403

               accuracy                           0.84     25976
              macro avg       0.84      0.84      0.84     25976
           weighted avg       0.84      0.84      0.84     25976

Confusion Matrix:
[[12453  2120]
 [ 2018  9385]]


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression Without Fuzzy,0.840699,0.815732,0.823029,0.819364


### **6.4 Logistic Regression dengan Fuzzy**

In [9]:
lr_hybrid_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

lr_hybrid_pipeline.fit(X_train_hybrid, y_train_hybrid)
y_pred_lr_hybrid = lr_hybrid_pipeline.predict(X_test_hybrid)

metrics_lr_hybrid = {
    "Model": "Logistic Regression With Fuzzy",
    "Accuracy": accuracy_score(y_test_hybrid, y_pred_lr_hybrid),
    "Precision": precision_score(y_test_hybrid, y_pred_lr_hybrid),
    "Recall": recall_score(y_test_hybrid, y_pred_lr_hybrid),
    "F1 Score": f1_score(y_test_hybrid, y_pred_lr_hybrid)
}

### **6.5 Hasil Logistic Regression dengan Fuzzy**

In [10]:
print("================= Logistic Regression With Fuzzy =================")

print("\nAccuracy :", round(metrics_lr_hybrid["Accuracy"], 4))
print("Precision:", round(metrics_lr_hybrid["Precision"], 4))
print("Recall   :", round(metrics_lr_hybrid["Recall"], 4))
print("F1 Score :", round(metrics_lr_hybrid["F1 Score"], 4))

print("\nClassification Report:")
print(classification_report(
    y_test_hybrid,
    y_pred_lr_hybrid,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_hybrid, y_pred_lr_hybrid))

display(pd.DataFrame([metrics_lr_hybrid]))

================= Logistic Regression With Fuzzy =================

Accuracy : 0.8442
Precision: 0.8189
Recall   : 0.8281
F1 Score : 0.8235

Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.86      0.86      0.86     14573
              satisfied       0.82      0.83      0.82     11403

               accuracy                           0.84     25976
              macro avg       0.84      0.84      0.84     25976
           weighted avg       0.84      0.84      0.84     25976

Confusion Matrix:
[[12485  2088]
 [ 1960  9443]]


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression With Fuzzy,0.844164,0.818923,0.828115,0.823494


### **6.6 Perbandingan Logistic Regression Tanpa Fuzzy dan dengan Fuzzy**

In [11]:
comparison_lr = pd.DataFrame([
    metrics_lr_original,
    metrics_lr_hybrid
])

comparison_delta_lr = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Without Fuzzy": [
        metrics_lr_original["Accuracy"],
        metrics_lr_original["Precision"],
        metrics_lr_original["Recall"],
        metrics_lr_original["F1 Score"]
    ],
    "With Fuzzy": [
        metrics_lr_hybrid["Accuracy"],
        metrics_lr_hybrid["Precision"],
        metrics_lr_hybrid["Recall"],
        metrics_lr_hybrid["F1 Score"]
    ]
})

comparison_delta_lr["Delta"] = (
    comparison_delta_lr["With Fuzzy"] - comparison_delta_lr["Without Fuzzy"]
)

display(comparison_delta_lr)

,Metric,Without Fuzzy,With Fuzzy,Delta
0,Accuracy,0.840699,0.844164,0.003465
1,Precision,0.815732,0.818923,0.003191
2,Recall,0.823029,0.828115,0.005086
3,F1 Score,0.819364,0.823494,0.004129


### **6.7 Kesimpulan Logistic Regression**

In [12]:
print("xxxxx")

xxxxx


## **7. Decision Tree**

### **7.1 Decision Tree Tanpa Fuzzy**

In [13]:
dt_original_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", DecisionTreeClassifier(random_state=42, max_depth=5))
    ]
)

dt_original_pipeline.fit(X_train_original, y_train_original)
y_pred_dt_original = dt_original_pipeline.predict(X_test_original)

metrics_dt_original = {
    "Model": "Decision Tree Without Fuzzy",
    "Accuracy": accuracy_score(y_test_original, y_pred_dt_original),
    "Precision": precision_score(y_test_original, y_pred_dt_original),
    "Recall": recall_score(y_test_original, y_pred_dt_original),
    "F1 Score": f1_score(y_test_original, y_pred_dt_original)
}

### **7.2 Visualisasi Decision Tree**

xxxxx

### **7.3 Hasil Decision Tree Tanpa Fuzzy**

In [14]:
print("================== Decision Tree Without Fuzzy ==================")
print("\nAccuracy :", round(metrics_dt_original["Accuracy"], 4))
print("Precision:", round(metrics_dt_original["Precision"], 4))
print("Recall   :", round(metrics_dt_original["Recall"], 4))
print("F1 Score :", round(metrics_dt_original["F1 Score"], 4))

print("\nClassification Report:")
print(classification_report(
    y_test_original,
    y_pred_dt_original,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_original, y_pred_dt_original))

display(pd.DataFrame([metrics_dt_original]))

================== Decision Tree Without Fuzzy ==================

Accuracy : 0.8968
Precision: 0.9176
Recall   : 0.8404
F1 Score : 0.8773

Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.88      0.94      0.91     14573
              satisfied       0.92      0.84      0.88     11403

               accuracy                           0.90     25976
              macro avg       0.90      0.89      0.89     25976
           weighted avg       0.90      0.90      0.90     25976

Confusion Matrix:
[[13712   861]
 [ 1820  9583]]


,Model,Accuracy,Precision,Recall,F1 Score
0,Decision Tree Without Fuzzy,0.896789,0.91756,0.840393,0.877283


### **7.4 Decision Tree dengan Fuzzy**

In [15]:
dt_hybrid_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", DecisionTreeClassifier(random_state=42, max_depth=5))
    ]
)

dt_hybrid_pipeline.fit(X_train_hybrid, y_train_hybrid)
y_pred_dt_hybrid = dt_hybrid_pipeline.predict(X_test_hybrid)

metrics_dt_hybrid = {
    "Model": "Decision Tree With Fuzzy",
    "Accuracy": accuracy_score(y_test_hybrid, y_pred_dt_hybrid),
    "Precision": precision_score(y_test_hybrid, y_pred_dt_hybrid),
    "Recall": recall_score(y_test_hybrid, y_pred_dt_hybrid),
    "F1 Score": f1_score(y_test_hybrid, y_pred_dt_hybrid)
}

### **7.5 Hasil Decision Tree dengan Fuzzy**

In [16]:
print("==================== Decision Tree With Fuzzy ====================")
print("\nAccuracy :", round(metrics_dt_hybrid["Accuracy"], 4))
print("Precision:", round(metrics_dt_hybrid["Precision"], 4))
print("Recall   :", round(metrics_dt_hybrid["Recall"], 4))
print("F1 Score :", round(metrics_dt_hybrid["F1 Score"], 4))

print("\nClassification Report:")
print(classification_report(
    y_test_hybrid,
    y_pred_dt_hybrid,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_hybrid, y_pred_dt_hybrid))

display(pd.DataFrame([metrics_dt_hybrid]))

==================== Decision Tree With Fuzzy ====================

Accuracy : 0.9055
Precision: 0.8918
Recall   : 0.8932
F1 Score : 0.8925

Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.92      0.92      0.92     14573
              satisfied       0.89      0.89      0.89     11403

               accuracy                           0.91     25976
              macro avg       0.90      0.90      0.90     25976
           weighted avg       0.91      0.91      0.91     25976

Confusion Matrix:
[[13337  1236]
 [ 1218 10185]]


,Model,Accuracy,Precision,Recall,F1 Score
0,Decision Tree With Fuzzy,0.905528,0.891778,0.893186,0.892482


### **7.6 Perbandingan Decision Tree tanpa Fuzzy dan dengan Fuzzy**

In [17]:
comparison_dt = pd.DataFrame([
    metrics_dt_original,
    metrics_dt_hybrid
])

comparison_delta_dt = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Without Fuzzy": [
        metrics_dt_original["Accuracy"],
        metrics_dt_original["Precision"],
        metrics_dt_original["Recall"],
        metrics_dt_original["F1 Score"]
    ],
    "With Fuzzy": [
        metrics_dt_hybrid["Accuracy"],
        metrics_dt_hybrid["Precision"],
        metrics_dt_hybrid["Recall"],
        metrics_dt_hybrid["F1 Score"]
    ]
})

comparison_delta_dt["Delta"] = (
    comparison_delta_dt["With Fuzzy"] - comparison_delta_dt["Without Fuzzy"]
)

display(comparison_delta_dt)

,Metric,Without Fuzzy,With Fuzzy,Delta
0,Accuracy,0.896789,0.905528,0.008739
1,Precision,0.917560,0.891778,-0.025782
2,Recall,0.840393,0.893186,0.052793
3,F1 Score,0.877283,0.892482,0.015199


### **7.7 Kesimpulan Decision Tree**

In [18]:
print("xxxxx")

xxxxx


## **8. Random Forest**

### **8.1 Random Forest Tanpa Fuzzy**

In [19]:
rf_original_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]
)

rf_original_pipeline.fit(X_train_original, y_train_original)
y_pred_rf_original = rf_original_pipeline.predict(X_test_original)

metrics_rf_original = {
    "Model": "Random Forest Without Fuzzy",
    "Accuracy": accuracy_score(y_test_original, y_pred_rf_original),
    "Precision": precision_score(y_test_original, y_pred_rf_original),
    "Recall": recall_score(y_test_original, y_pred_rf_original),
    "F1 Score": f1_score(y_test_original, y_pred_rf_original)
}

### **8.2 Visualisasi Random Forest**

xxxxx

### **8.3 Hasil Random Forest Tanpa Fuzzy**

In [20]:
print("================== Random Forest Without Fuzzy ==================")
print("\nAccuracy :", round(metrics_rf_original["Accuracy"], 4))
print("Precision:", round(metrics_rf_original["Precision"], 4))
print("Recall   :", round(metrics_rf_original["Recall"], 4))
print("F1 Score :", round(metrics_rf_original["F1 Score"], 4))

print("\nClassification Report:")
print(classification_report(
    y_test_original,
    y_pred_rf_original,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_original, y_pred_rf_original))

display(pd.DataFrame([metrics_rf_original]))

================== Random Forest Without Fuzzy ==================

Accuracy : 0.9253
Precision: 0.9293
Recall   : 0.8982
F1 Score : 0.9135

Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.92      0.95      0.93     14573
              satisfied       0.93      0.90      0.91     11403

               accuracy                           0.93     25976
              macro avg       0.93      0.92      0.92     25976
           weighted avg       0.93      0.93      0.93     25976

Confusion Matrix:
[[13794   779]
 [ 1161 10242]]


,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest Without Fuzzy,0.925316,0.929317,0.898185,0.913486


### **8.4 Random Forest dengan Fuzzy**

In [21]:
rf_hybrid_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]
)

rf_hybrid_pipeline.fit(X_train_hybrid, y_train_hybrid)
y_pred_rf_hybrid = rf_hybrid_pipeline.predict(X_test_hybrid)

metrics_rf_hybrid = {
    "Model": "Random Forest With Fuzzy",
    "Accuracy": accuracy_score(y_test_hybrid, y_pred_rf_hybrid),
    "Precision": precision_score(y_test_hybrid, y_pred_rf_hybrid),
    "Recall": recall_score(y_test_hybrid, y_pred_rf_hybrid),
    "F1 Score": f1_score(y_test_hybrid, y_pred_rf_hybrid)
}

### **8.5 Hasil Random Forest dengan Fuzzy**

In [22]:
print("=================== Random Forest With Fuzzy ===================")
print("\nAccuracy :", round(metrics_rf_hybrid["Accuracy"], 4))
print("Precision:", round(metrics_rf_hybrid["Precision"], 4))
print("Recall   :", round(metrics_rf_hybrid["Recall"], 4))
print("F1 Score :", round(metrics_rf_hybrid["F1 Score"], 4))

print("Classification Report:")
print(classification_report(
    y_test_hybrid,
    y_pred_rf_hybrid,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_hybrid, y_pred_rf_hybrid))

display(pd.DataFrame([metrics_rf_hybrid]))

=================== Random Forest With Fuzzy ===================

Accuracy : 0.9253
Precision: 0.9293
Recall   : 0.8982
F1 Score : 0.9135
Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.92      0.95      0.93     14573
              satisfied       0.93      0.90      0.91     11403

               accuracy                           0.93     25976
              macro avg       0.93      0.92      0.92     25976
           weighted avg       0.93      0.93      0.93     25976

Confusion Matrix:
[[13794   779]
 [ 1161 10242]]


,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest With Fuzzy,0.925316,0.929317,0.898185,0.913486


### **8.6 Perbandingan Random Forest Tanpa Fuzzy dan dengan Fuzzy**

In [23]:
comparison_rf = pd.DataFrame([
    metrics_rf_original,
    metrics_rf_hybrid
])

comparison_delta_rf = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Without Fuzzy": [
        metrics_rf_original["Accuracy"],
        metrics_rf_original["Precision"],
        metrics_rf_original["Recall"],
        metrics_rf_original["F1 Score"]
    ],
    "With Fuzzy": [
        metrics_rf_hybrid["Accuracy"],
        metrics_rf_hybrid["Precision"],
        metrics_rf_hybrid["Recall"],
        metrics_rf_hybrid["F1 Score"]
    ]
})

comparison_delta_rf["Delta"] = (
    comparison_delta_rf["With Fuzzy"] - comparison_delta_rf["Without Fuzzy"]
)

display(comparison_delta_rf)

,Metric,Without Fuzzy,With Fuzzy,Delta
0,Accuracy,0.925316,0.925316,0.0
1,Precision,0.929317,0.929317,0.0
2,Recall,0.898185,0.898185,0.0
3,F1 Score,0.913486,0.913486,0.0


### **8.7 Kesimpulan Random Forest**

In [24]:
print("xxxxx")

xxxxx


## **9. XGBoost**

### **9.1 XGBoost Tanpa Fuzzy**

In [25]:
xgb_original_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss', n_jobs=-1))
    ]
)

xgb_original_pipeline.fit(X_train_original, y_train_original)
y_pred_xgb_original = xgb_original_pipeline.predict(X_test_original)

metrics_xgb_original = {
    "Model": "XGBoost Without Fuzzy",
    "Accuracy": accuracy_score(y_test_original, y_pred_xgb_original),
    "Precision": precision_score(y_test_original, y_pred_xgb_original),
    "Recall": recall_score(y_test_original, y_pred_xgb_original),
    "F1 Score": f1_score(y_test_original, y_pred_xgb_original)
}

### **9.2 Visualisasi XGBoost**

xxxxx

### **9.3 Hasil XGBoost Tanpa Fuzzy**

In [26]:
print("===================== XGBoost Without Fuzzy =====================")
print("\nAccuracy :", round(metrics_xgb_original["Accuracy"], 4))
print("Precision:", round(metrics_xgb_original["Precision"], 4))
print("Recall   :", round(metrics_xgb_original["Recall"], 4))
print("F1 Score :", round(metrics_xgb_original["F1 Score"], 4))

print("\nClassification Report:")
print(classification_report(
    y_test_original,
    y_pred_xgb_original,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_original, y_pred_xgb_original))

display(pd.DataFrame([metrics_xgb_original]))

===================== XGBoost Without Fuzzy =====================

Accuracy : 0.9219
Precision: 0.9267
Recall   : 0.8926
F1 Score : 0.9093

Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.92      0.94      0.93     14573
              satisfied       0.93      0.89      0.91     11403

               accuracy                           0.92     25976
              macro avg       0.92      0.92      0.92     25976
           weighted avg       0.92      0.92      0.92     25976

Confusion Matrix:
[[13768   805]
 [ 1225 10178]]


,Model,Accuracy,Precision,Recall,F1 Score
0,XGBoost Without Fuzzy,0.921851,0.926705,0.892572,0.909318


### **9.4 XGBoost dengan Fuzzy**

In [27]:
xgb_hybrid_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
                ],
                remainder="passthrough"
            )
        ),
        ("classifier", XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss', n_jobs=-1))
    ]
)

xgb_hybrid_pipeline.fit(X_train_hybrid, y_train_hybrid)
y_pred_xgb_hybrid = xgb_hybrid_pipeline.predict(X_test_hybrid)

metrics_xgb_hybrid = {
    "Model": "XGBoost With Fuzzy",
    "Accuracy": accuracy_score(y_test_hybrid, y_pred_xgb_hybrid),
    "Precision": precision_score(y_test_hybrid, y_pred_xgb_hybrid),
    "Recall": recall_score(y_test_hybrid, y_pred_xgb_hybrid),
    "F1 Score": f1_score(y_test_hybrid, y_pred_xgb_hybrid)
}

### **9.5 Hasil XGBoost dengan Fuzzy**

In [28]:
print("====================== XGBoost With Fuzzy ======================")
print("Accuracy :", round(metrics_xgb_hybrid["Accuracy"], 4))
print("Precision:", round(metrics_xgb_hybrid["Precision"], 4))
print("Recall   :", round(metrics_xgb_hybrid["Recall"], 4))
print("F1 Score :", round(metrics_xgb_hybrid["F1 Score"], 4))

print("\nClassification Report:")
print(classification_report(
    y_test_hybrid,
    y_pred_xgb_hybrid,
    target_names=label_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_hybrid, y_pred_xgb_hybrid))

display(pd.DataFrame([metrics_xgb_hybrid]))

====================== XGBoost With Fuzzy ======================
Accuracy : 0.9252
Precision: 0.9295
Recall   : 0.8977
F1 Score : 0.9133

Classification Report:
                         precision    recall  f1-score   support

neutral or dissatisfied       0.92      0.95      0.93     14573
              satisfied       0.93      0.90      0.91     11403

               accuracy                           0.93     25976
              macro avg       0.93      0.92      0.92     25976
           weighted avg       0.93      0.93      0.93     25976

Confusion Matrix:
[[13797   776]
 [ 1167 10236]]


,Model,Accuracy,Precision,Recall,F1 Score
0,XGBoost With Fuzzy,0.9252,0.929531,0.897659,0.913317


### **9.6 Perbandingan XGBoost Tanpa Fuzzy dan dengan Fuzzy**

In [29]:
comparison_xgb = pd.DataFrame([
    metrics_xgb_original,
    metrics_xgb_hybrid
])

comparison_delta_xgb = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Without Fuzzy": [
        metrics_xgb_original["Accuracy"],
        metrics_xgb_original["Precision"],
        metrics_xgb_original["Recall"],
        metrics_xgb_original["F1 Score"]
    ],
    "With Fuzzy": [
        metrics_xgb_hybrid["Accuracy"],
        metrics_xgb_hybrid["Precision"],
        metrics_xgb_hybrid["Recall"],
        metrics_xgb_hybrid["F1 Score"]
    ]
})

comparison_delta_xgb["Delta"] = (
    comparison_delta_xgb["With Fuzzy"] - comparison_delta_xgb["Without Fuzzy"]
)

display(comparison_delta_xgb)

,Metric,Without Fuzzy,With Fuzzy,Delta
0,Accuracy,0.921851,0.925200,0.003349
1,Precision,0.926705,0.929531,0.002827
2,Recall,0.892572,0.897659,0.005086
3,F1 Score,0.909318,0.913317,0.003999


### **9.7 Kesimpulan XGBoost**

In [30]:
print("xxxxx")

xxxxx


## **10. Rekap Semua Model**

In [31]:
comparison_all_models = pd.DataFrame([
    metrics_lr_original,
    metrics_lr_hybrid,
    metrics_dt_original,
    metrics_dt_hybrid,
    metrics_rf_original,
    metrics_rf_hybrid,
    metrics_xgb_original,
    metrics_xgb_hybrid
])

comparison_all_models = comparison_all_models[
    ["Model", "Accuracy", "Precision", "Recall", "F1 Score"]
]

display(comparison_all_models.sort_values(by="Accuracy", ascending=False))

,Model,Accuracy,Precision,Recall,F1 Score
5,Random Forest With Fuzzy,0.925316,0.929317,0.898185,0.913486
4,Random Forest Without Fuzzy,0.925316,0.929317,0.898185,0.913486
7,XGBoost With Fuzzy,0.925200,0.929531,0.897659,0.913317
6,XGBoost Without Fuzzy,0.921851,0.926705,0.892572,0.909318
3,Decision Tree With Fuzzy,0.905528,0.891778,0.893186,0.892482
2,Decision Tree Without Fuzzy,0.896789,0.917560,0.840393,0.877283
1,Logistic Regression With Fuzzy,0.844164,0.818923,0.828115,0.823494
0,Logistic Regression Without Fuzzy,0.840699,0.815732,0.823029,0.819364


## **11. Kesimpulan Machine Learning**

In [32]:
print("xxxxx")

xxxxx
